In [ ]:
# =================================================================================
# SCRIPT ZUM ÜBERWACHTEN TRAINING EINES REGISTRIERUNGSNETZWERKS
# =================================================================================
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cupy as np
import zarr
import tqdm

# --- 1. Die Modell-Architektur (NUR das U-Net wird als Modell benötigt) ---
class UNet3D(nn.Module):
    # ... (Die komplette UNet3D-Klasse von vorher hier einfügen) ...
    def __init__(self, in_channels=2, out_channels=3):
        super().__init__()
        self.enc1 = self._conv_block(in_channels, 16)
        self.enc2 = self._conv_block(16, 32)
        self.pool = nn.MaxPool3d(2)
        self.bottleneck = self._conv_block(32, 64)
        self.upconv2 = nn.ConvTranspose3d(64, 32, kernel_size=2, stride=2)
        self.dec2 = self._conv_block(64, 32)
        self.upconv1 = nn.ConvTranspose3d(32, 16, kernel_size=2, stride=2)
        self.dec1 = self._conv_block(32, 16)
        self.final_conv = nn.Conv3d(16, out_channels, kernel_size=1)
        self.final_conv.weight.data.zero_()
        self.final_conv.bias.data.zero_()
    def _conv_block(self, in_c, out_c):
        return nn.Sequential(nn.Conv3d(in_c, out_c, 3, 1, 1), nn.ReLU(True), nn.Conv3d(out_c, out_c, 3, 1, 1), nn.ReLU(True))
    def forward(self, x_fixed, x_moving):
        x = torch.cat([x_fixed, x_moving], dim=1)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.upconv2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.upconv1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        return self.final_conv(d1)


# --- 2. Der neue Dataset-Loader für überwachtes Training ---
class SupervisedDisplacementDataset(Dataset):
    # Ersetzen Sie nur die __init__-Methode in Ihrer Dataset-Klasse.

    def __init__(self, moving_path, fixed_path, dvf_gt_path):
        super().__init__()
        self.moving_arr = zarr.open(moving_path, mode='r')
        self.fixed_arr = zarr.open(fixed_path, mode='r')
        self.dvf_gt_arr = zarr.open(dvf_gt_path, mode='r')
        
        # --- KORRIGIERTE ASSERT-ANWEISUNG ---
        # Wir vergleichen die Zeit-Dimension (immer an Index 3) für alle Arrays.
        assert self.moving_arr.shape[3] == self.fixed_arr.shape[3] == self.dvf_gt_arr.shape[3], \
            (f"Anzahl der Bilder/DVFs stimmt nicht überein! "
             f"Moving: {self.moving_arr.shape[3]}, "
             f"Fixed: {self.fixed_arr.shape[3]}, "
             f"DVF: {self.dvf_gt_arr.shape[3]}")
    
        self.num_images = self.moving_arr.shape[3]
    
        # Padding-Logik (unverändert)
        self.original_shape = self.moving_arr.shape
        self.padded_shape = [s for s in self.original_shape[:3]]
        for i in range(3):
            if self.padded_shape[i] % 4 != 0:
                self.padded_shape[i] = (self.padded_shape[i] // 4 + 1) * 4
    
    def __len__(self):
        return self.num_images

    # Ersetzen Sie nur die __getitem__-Methode in Ihrer Dataset-Klasse.
    # Der Rest der Klasse (__init__, __len__, _preprocess_image, _pad_tensor) bleibt gleich.
    
    def __getitem__(self, idx):
        # Lade Bilder (unverändert)
        moving_np = self.moving_arr[..., idx]
        fixed_np = self.fixed_arr[..., idx]
        
        # --- KORRIGIERTE LADE- UND UMFORMUNGSLOGIK FÜR DAS DVF ---
    
        # 1. KORREKTES SLICING: Lade das DVF für den Zeitpunkt `idx`
        # Wir slicen explizit die 4. Dimension (Index 3), die der Zeit entspricht.
        dvf_gt_np = self.dvf_gt_arr[:, :, :, idx, :] # Ergibt ein Array der Form (H, W, D, 3)
    
        # 2. KORREKTE UMFORMUNG: Konvertiere von (H, W, D, 3) zu (3, D, H, W) für PyTorch
        # Achsen: 0=H, 1=W, 2=D, 3=Vektor
        # Ziel:   0=Vektor, 1=D, 2=H, 3=W  => transpose(3, 2, 0, 1)
        dvf_gt_tensor = torch.from_numpy(dvf_gt_np.astype(np.float32)).permute(3, 2, 0, 1)
    
        # Vorverarbeitung (dieser Teil bleibt unverändert)
        moving_tensor = self._preprocess_image(moving_np)
        fixed_tensor = self._preprocess_image(fixed_np)
        
        # Padding auf das DVF anwenden (dieser Teil bleibt unverändert)
        dvf_gt_tensor = self._pad_tensor(dvf_gt_tensor)
    
        return moving_tensor, fixed_tensor, dvf_gt_tensor
    
    def _preprocess_image(self, volume_np):
        # Hier deine Normalisierungslogik einfügen, falls gewünscht
        tensor = torch.from_numpy(volume_np.astype(np.float32)).permute(2, 0, 1).unsqueeze(0)
        return self._pad_tensor(tensor)

    def _pad_tensor(self, tensor):
        # Nimmt einen Tensor der Form (C,D,H,W)
        pad_d = self.padded_shape[2] - tensor.shape[1]
        pad_h = self.padded_shape[0] - tensor.shape[2]
        pad_w = self.padded_shape[1] - tensor.shape[3]
        padding = (pad_w // 2, pad_w - pad_w // 2, pad_h // 2, pad_h - pad_h // 2, pad_d // 2, pad_d - pad_d // 2)
        return F.pad(tensor, padding, "constant", 0)

# --- 3. Das neue, überwachte Trainings-Skript ---
if __name__ == '__main__':
    # --- Konfiguration ---
    MODEL_PATH = "/media/shooty/Dev/repos/MRI-MoCoCo/supervised_model_1.pth"
    MOVING_PATH = "MRI-Datasets/DCE"
    FIXED_PATH = "MRI-Datasets/mdreg_DCE_fitting_results/coreg_zarr_2.zarr"
    DVF_GT_PATH = "MRI-Datasets/mdreg_DCE_fitting_results/transfo_zarr_2.zarr" # <-- WICHTIG: Pfad zu den Ground-Truth-Daten
    MODEL_SAVE_PATH = "./supervised_model_1.pth"
    BATCH_SIZE = 2
    LEARNING_RATE = 1e-5
    NUM_EPOCHS = 100 # Starte mit mehr Epochen

    # --- Setup ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dataset = SupervisedDisplacementDataset(MOVING_PATH, FIXED_PATH, DVF_GT_PATH)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    
    # --- Modell, Optimizer und Loss-Funktion ---
    model = UNet3D().to(device) # Das U-Net ist jetzt unser vollständiges Modell
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.MSELoss() # Mean Squared Error ist der Standard für Regressionsprobleme
    
    # --- Trainings-Schleife ---
    print("Starte überwachtes Training...")
    for epoch in range(NUM_EPOCHS):
        epoch_loss = 0.0
        for moving_batch, fixed_batch, dvf_gt_batch in tqdm.tqdm(dataloader, desc=f"Epoche {epoch+1}"):
            moving_batch = moving_batch.to(device)
            fixed_batch = fixed_batch.to(device)
            dvf_gt_batch = dvf_gt_batch.to(device)

            optimizer.zero_grad()
            
            # Forward-Pass: Das Modell sagt das DVF voraus
            predicted_dvf = model(fixed_batch, moving_batch)
            
            # Verlust berechnen: Vergleiche Vorhersage mit der Wahrheit
            loss = loss_fn(predicted_dvf, dvf_gt_batch)
            
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
        print(f"Epoche {epoch+1} - Durchschnittlicher MSE Loss: {epoch_loss / len(dataloader):.6f}")

    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"✔️ Training abgeschlossen. Modell gespeichert unter: {MODEL_SAVE_PATH}")

In [ ]:
import os
import time

print("Alle Berechnungen sind abgeschlossen.")
print("Der Computer wird in 60 Sekunden heruntergefahren...")
print("Drücken Sie Strg+C in der Konsole, in der Jupyter läuft, um abzubrechen.")

# Eine kleine Wartezeit, um den Vorgang ggf. noch abbrechen zu können
time.sleep(60) 

# Der Befehl zum Herunterfahren
# 'sudo' ist nötig, aber dank der Konfiguration wird kein Passwort benötigt.
# 'now' bedeutet, dass der PC sofort heruntergefahren wird.
shutdown_command = "sudo shutdown now"

print("Sende Befehl zum Herunterfahren...")
try:
    os.system(shutdown_command)
except Exception as e:
    print(f"Fehler beim Herunterfahren: {e}")
    print("Möglicherweise müssen die sudo-Rechte wie beschrieben konfiguriert werden.")

Alle Berechnungen sind abgeschlossen.
Der Computer wird in 60 Sekunden heruntergefahren...
Drücken Sie Strg+C in der Konsole, in der Jupyter läuft, um abzubrechen.
